# mplstudio × scanpy

mplstudio detects the two most common scanpy plot styles and adds a **`palette=`** kwarg to carry scanpy's colour assignments directly into the GUI.

| Pattern | What scanpy does | What mplstudio does |
|---|---|---|
| Categorical UMAP | One `PathCollection` per category, no `ax.legend()` | Detects unlabeled collections, falls back to direct scan |
| Continuous UMAP | Single `PathCollection` + `set_array()`, label `""` | Picks up scalar-mapped collection, shows colormap controls |

**Install:**
```bash
pip install mplstudio scanpy
```

The notebook runs fine without scanpy too — it falls back to synthetic data that reproduces the same matplotlib patterns.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mplstudio

try:
    import scanpy as sc
    HAS_SCANPY = True
    print(f"scanpy {sc.__version__} found — using real AnnData")
except ImportError:
    HAS_SCANPY = False
    print("scanpy not installed — using synthetic data that mirrors scanpy's matplotlib output")

## 1. Categorical UMAP

Scanpy draws one `PathCollection` per category and no `ax.legend()` (it renders its own external legend via `AnchoredOffsetbox`).  
mplstudio's fallback scan picks up all labeled collections automatically.

In [ ]:
# Palette that mirrors what would live in adata.uns["cell_type_colors"]
CELL_TYPE_COLORS = {
    "T cell":    "#e41a1c",
    "B cell":    "#377eb8",
    "NK cell":   "#4daf4a",
    "Monocyte":  "#ff7f00",
    "DC":        "#984ea3",
}

if HAS_SCANPY:
    # ── real scanpy path ──────────────────────────────────────────────────
    adata = sc.datasets.pbmc68k_reduced()
    sc.pp.neighbors(adata)
    sc.tl.umap(adata)

    fig_cat, ax_cat = plt.subplots(figsize=(5, 4))
    sc.pl.umap(adata, color="bulk_labels", ax=ax_cat, show=False)
    # adata.uns["bulk_labels_colors"] holds the per-category hex strings
    palette = dict(zip(
        adata.obs["bulk_labels"].cat.categories,
        adata.uns["bulk_labels_colors"],
    ))
    plt.close()

else:
    # ── synthetic fallback: same matplotlib structure as scanpy ───────────
    rng = np.random.default_rng(0)

    # Simulate UMAP 2-D embedding per cell type
    centres = {"T cell": (2, 1), "B cell": (-2, 1), "NK cell": (0, -2),
               "Monocyte": (2, -1), "DC": (-2, -1)}

    fig_cat, ax_cat = plt.subplots(figsize=(5, 4))
    for ct, (cx, cy) in centres.items():
        xy = rng.standard_normal((80, 2)) * 0.6 + [cx, cy]
        # scanpy: one PathCollection per category, label=category name
        ax_cat.scatter(xy[:, 0], xy[:, 1],
                       c=CELL_TYPE_COLORS[ct], label=ct, s=8, alpha=0.85)
    # scanpy does NOT call ax.legend() here
    ax_cat.set_xlabel("UMAP1")
    ax_cat.set_ylabel("UMAP2")
    ax_cat.set_title("UMAP — cell type (synthetic)")
    plt.close()

    palette = CELL_TYPE_COLORS.copy()

print("Palette:", palette)

In [ ]:
# Pass the palette so mplstudio shows a "Custom" colour mode
# that pre-fills each category with its original scanpy colour.
mplstudio.studio(fig_cat, palette=palette)

**What to try in the panel:**

- **Colors → Custom**: restores the original scanpy palette at any time.
- **Colors → Manual**: individual pickers are pre-filled from the palette — tweak single categories.
- **Colors → Palette / Smart**: switch to a built-in palette or a perceptually-uniform auto palette.
- **Typography**: adjust font size and family for axis labels / title.
- **Axes**: tighten axis limits or add/remove grid.

## 2. Continuous UMAP (gene expression)

Scanpy plots gene expression as a single `PathCollection` with `set_array()` (per-cell values) and label `""`.  
mplstudio detects this as a *continuous* plot and shows the colormap picker.

In [ ]:
if HAS_SCANPY:
    fig_cont, ax_cont = plt.subplots(figsize=(5, 4))
    gene = adata.var_names[0]  # first gene
    sc.pl.umap(adata, color=gene, ax=ax_cont, show=False)
    plt.close()

else:
    rng2 = np.random.default_rng(1)
    n = 400
    angle = rng2.uniform(0, 2 * np.pi, n)
    radius = rng2.uniform(0, 3, n)
    x = radius * np.cos(angle)
    y = radius * np.sin(angle)
    # Gene expression: higher near the centre
    expr = np.exp(-radius / 1.5) + rng2.standard_normal(n) * 0.05
    expr = np.clip(expr, 0, None)

    fig_cont, ax_cont = plt.subplots(figsize=(5, 4))
    # scanpy: single scatter, label="", c=array via set_array internally
    scat = ax_cont.scatter(x, y, c=expr, cmap="viridis", s=8, vmin=0)
    fig_cont.colorbar(scat, ax=ax_cont, label="expr (log-norm)")
    ax_cont.set_xlabel("UMAP1")
    ax_cont.set_ylabel("UMAP2")
    ax_cont.set_title("UMAP — gene expression (synthetic)")
    plt.close()

mplstudio.studio(fig_cont)

**What to try:**

- **Colors → Sequential**: swap `viridis` for `plasma`, `inferno`, or `magma`.
- **Colors → Diverging**: try `RdBu_r` or `coolwarm` to highlight cells above / below the mean.
- **Save**: export a publication-ready PNG at 300 dpi.

## 3. Mixed figure — categorical overlay on continuous background

Sometimes you want to overlay labelled scatter groups on top of a density contour or heatmap.  
mplstudio detects **both** and shows the full palette + colormap controls together.

In [ ]:
rng3 = np.random.default_rng(2)

# Continuous background: 2-D KDE-like density (contourf)
xg = np.linspace(-4, 4, 80)
yg = np.linspace(-4, 4, 80)
Xg, Yg = np.meshgrid(xg, yg)
density = (
    np.exp(-((Xg - 1.5)**2 + (Yg - 1.0)**2) / 2.5) +
    np.exp(-((Xg + 1.5)**2 + (Yg - 0.5)**2) / 2.0) * 0.7
)

fig_mix, ax_mix = plt.subplots(figsize=(5, 4))
cf = ax_mix.contourf(Xg, Yg, density, levels=10, cmap="Blues", alpha=0.6)
fig_mix.colorbar(cf, ax=ax_mix, label="cell density")

# Categorical overlay: two annotated populations
for name, cx, cy, col in [("Pop A", 1.5, 1.0, "#E69F00"),
                           ("Pop B", -1.5, 0.5, "#56B4E9")]:
    xy = rng3.standard_normal((50, 2)) * 0.5 + [cx, cy]
    ax_mix.scatter(xy[:, 0], xy[:, 1], label=name, color=col,
                   edgecolors="white", linewidths=0.4, s=30)
ax_mix.legend()
ax_mix.set_title("Density + categorical overlay")
ax_mix.set_xlabel("UMAP1")
ax_mix.set_ylabel("UMAP2")
plt.close()

mplstudio.studio(fig_mix)

## 4. Programmatic style API (no GUI)

All style functions are accessible directly from `mplstudio.style` — useful for scripted figure generation.

In [ ]:
from mplstudio import style as S

# Rebuild the categorical figure and apply scanpy palette programmatically
rng4 = np.random.default_rng(0)
centres = {"T cell": (2, 1), "B cell": (-2, 1), "NK cell": (0, -2),
           "Monocyte": (2, -1), "DC": (-2, -1)}

fig_api, ax_api = plt.subplots(figsize=(5, 4))
for ct, (cx, cy) in centres.items():
    xy = rng4.standard_normal((80, 2)) * 0.6 + [cx, cy]
    ax_api.scatter(xy[:, 0], xy[:, 1], label=ct, s=8, alpha=0.85)
ax_api.set_title("Before styling")

# Apply scanpy palette via the style API
labels  = S.get_line_labels(fig_api)
colors  = [CELL_TYPE_COLORS.get(lbl, "#888888") for lbl in labels]
S.set_line_colors_manual(fig_api, colors)
S.set_font_size(fig_api, 12)
S.set_spine_style(fig_api, "2-Side")

ax_api.set_title("After styling")
plt.tight_layout()
plt.show()